# 02 — Project Functions

This notebook contains reusable functions for the Vanguard project.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import ttest_ind

pd.set_option("display.max_columns", None)


## Load clean data

In [ ]:

def load_clean_data(path="processed_data/clean_vanguard_data.csv"):
    """
    Loads the cleaned Vanguard dataset.
    """
    return pd.read_csv(path)


## Add step numbers

In [ ]:

def add_step_numbers(df):
    """
    Adds numeric process step order.
    """
    df = df.copy()

    step_mapping = {
        "start": 0,
        "step_1": 1,
        "step_2": 2,
        "step_3": 3,
        "confirm": 4
    }

    df["process_step"] = df["process_step"].str.lower().str.strip()
    df["step_num"] = df["process_step"].map(step_mapping)

    return df


## Create session data

In [ ]:

def create_session_data(df):
    """
    Creates session-level data from event-level activity.
    """
    df = df.copy()

    df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")

    if "step_num" not in df.columns:
        df = add_step_numbers(df)

    session_data = (
        df
        .groupby(["variation", "client_id", "visitor_id", "visit_id"], as_index=False)
        .agg(
            max_step=("step_num", "max"),
            first_timestamp=("date_time", "min"),
            last_timestamp=("date_time", "max"),
            number_of_steps=("process_step", "count")
        )
    )

    session_data["completed"] = session_data["max_step"] == 4

    session_data["session_duration_seconds"] = (
        session_data["last_timestamp"] - session_data["first_timestamp"]
    ).dt.total_seconds()

    return session_data


## Completion rate

In [ ]:

def calculate_completion_rate(session_data):
    """
    Calculates completion rate by variation.
    """
    summary = (
        session_data
        .groupby("variation")
        .agg(
            total_sessions=("visit_id", "count"),
            completed_sessions=("completed", "sum")
        )
        .reset_index()
    )

    summary["completion_rate"] = (
        summary["completed_sessions"] / summary["total_sessions"]
    )

    summary["completion_rate_percent"] = summary["completion_rate"] * 100

    return summary


## Error rate

In [ ]:

def calculate_error_rate(df):
    """
    Calculates error rate using backward movement between steps.
    """
    df = df.copy()

    df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")

    if "step_num" not in df.columns:
        df = add_step_numbers(df)

    df = df.sort_values(
        by=["client_id", "visitor_id", "visit_id", "date_time"]
    )

    df["previous_step"] = (
        df
        .groupby(["client_id", "visitor_id", "visit_id"])["step_num"]
        .shift(1)
    )

    df["is_error"] = df["step_num"] < df["previous_step"]

    error_summary = (
        df
        .groupby("variation")
        .agg(
            total_actions=("process_step", "count"),
            total_errors=("is_error", "sum")
        )
        .reset_index()
    )

    error_summary["error_rate"] = (
        error_summary["total_errors"] / error_summary["total_actions"]
    )

    error_summary["error_rate_percent"] = error_summary["error_rate"] * 100

    return error_summary


## Average session duration

In [ ]:

def calculate_average_session_duration(session_data):
    """
    Calculates average and median session duration.
    """
    return (
        session_data
        .groupby("variation")
        .agg(
            average_duration_seconds=("session_duration_seconds", "mean"),
            median_duration_seconds=("session_duration_seconds", "median"),
            session_count=("session_duration_seconds", "count")
        )
        .reset_index()
    )


## Funnel summary

In [ ]:

def calculate_funnel_summary(df):
    """
    Calculates how many sessions reached each step.
    """
    df = df.copy()

    if "step_num" not in df.columns:
        df = add_step_numbers(df)

    session_steps = (
        df
        .groupby(["variation", "client_id", "visitor_id", "visit_id"], as_index=False)
        .agg(max_step=("step_num", "max"))
    )

    funnel_rows = []

    for variation in session_steps["variation"].unique():
        temp = session_steps[session_steps["variation"] == variation]
        total_sessions = len(temp)

        for step_num in range(5):
            reached_step = (temp["max_step"] >= step_num).sum()

            funnel_rows.append({
                "variation": variation,
                "step_num": step_num,
                "sessions_reached": reached_step,
                "total_sessions": total_sessions,
                "step_rate": reached_step / total_sessions
            })

    funnel = pd.DataFrame(funnel_rows)

    step_names = {
        0: "start",
        1: "step_1",
        2: "step_2",
        3: "step_3",
        4: "confirm"
    }

    funnel["step_name"] = funnel["step_num"].map(step_names)
    funnel["step_rate_percent"] = funnel["step_rate"] * 100

    return funnel


## Two-proportion z-test

In [ ]:

def run_completion_ztest(summary):
    """
    Runs a two-proportion z-test for completion rate.
    """
    control = summary[summary["variation"] == "control"]
    test = summary[summary["variation"] == "test"]

    n_control = control["total_sessions"].values[0]
    n_test = test["total_sessions"].values[0]

    x_control = control["completed_sessions"].values[0]
    x_test = test["completed_sessions"].values[0]

    z_stat, p_value = proportions_ztest(
        [x_control, x_test],
        [n_control, n_test]
    )

    control_rate = x_control / n_control
    test_rate = x_test / n_test
    lift = test_rate - control_rate

    return {
        "z_stat": z_stat,
        "p_value": p_value,
        "control_rate": control_rate,
        "test_rate": test_rate,
        "lift": lift
    }


## Plot completion rate

In [ ]:

def plot_completion_rate(summary):
    """
    Plots completion rate by variation.
    """
    plt.figure(figsize=(8, 5))
    plt.bar(summary["variation"], summary["completion_rate_percent"])

    plt.title("Completion Rate by Variation")
    plt.xlabel("Variation")
    plt.ylabel("Completion Rate (%)")
    plt.ylim(0, 100)
    plt.show()


## Quick test

In [ ]:

df = load_clean_data()
df = add_step_numbers(df)

session_data = create_session_data(df)

completion_summary = calculate_completion_rate(session_data)
display(completion_summary)

error_summary = calculate_error_rate(df)
display(error_summary)

funnel_summary = calculate_funnel_summary(df)
display(funnel_summary.head())
